# L12b Lab: Q-Learning for Grid World Navigation
In this lab, you will implement Q-learning for a robot on a rectangular grid that must reach a charging station while avoiding lava pits, without being told where either one is.

__Backstory:__ Previously, we guided the robot to its charging station with value iteration, which needs the reward of every move and the probability of every next square. A real robot is not handed these: it finds out what a move does only by making it. Your job is to write Q-learning, let the robot learn the world by moving around in it, and compare the policy it learns with the optimal policy from value iteration.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Implement tabular Q-learning:__ Q-learning keeps a table of action values and updates one entry after each move, using only the reward and the next square that the world returns. We'll complete our own version and check it on small problems whose answers we know.
> * __Learn a path through the lava grid world:__ The robot starts with a table of zeros and learns from many episodes of trial and error. We'll train it and draw the path its learned policy takes to the charging station.
> * __Compare the learned policy with the optimal one:__ Value iteration computes the optimal policy because it knows the rewards and the transition probabilities. We'll count the squares where the learned move is optimal and see how the count changes with the training.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your Q-learning function goes in [src/Compute.jl](src/Compute.jl). We call [the `L12bQLearning.myqlearning(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.
___

## Task 1: Build the Lava Grid World
In this task, we build the lava grid world and a function that lets the robot try a move and see what happens.

We use the same lava grid world as before: a 20 by 20 grid with a charging station and five lava pits. We represent it with [the `MyRectangularGridWorldModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.MyRectangularGridWorldModel), which maps $(x,y)$ coordinates onto discrete states.

> __Grid World Structure:__ The model holds:
> * a rectangular grid, with one state per square
> * a reward for reaching the charging station, and a penalty for a lava pit
> * a small cost for every ordinary move, which rewards a short path
> * absorbing states, the charging station and the lava pits: once the robot enters one, the episode ends

Let's set the grid dimensions, the number of actions, and the discount factor $\gamma$ that weighs future rewards against immediate ones:

In [ ]:
number_of_rows = 20; # number of rows in the grid world
number_of_cols = 20; # number of cols in the grid world
nactions = 4; # number of actions: left, right, down, up
γ = 0.95; # discount factor
nstates = (number_of_rows*number_of_cols); # number of states, one for each square
𝒮 = range(1,stop=nstates,step=1) |> collect; # states
𝒜 = range(1,stop=nactions,step=1) |> collect; # actions

Next we describe what each square is worth, in the `rewards::Dict{Tuple{Int,Int}, Float64}` dictionary, which maps an $(x,y)$ coordinate to a reward, and the `absorbing_state_set::Set{Tuple{Int,Int}}` set, which collects the squares that end the episode. Only the squares that differ from an ordinary move appear here:

In [ ]:
# setup rewards -
lava_reward = -1000.0;
charging_reward = 100.0;

rewards = Dict{Tuple{Int,Int}, Float64}()
rewards[(1,2)] = lava_reward # lava in the (1,2) square
rewards[(2,2)] = lava_reward # lava in the (2,2) square
rewards[(2,3)] = lava_reward # lava in the (2,3) square
rewards[(4,5)] = lava_reward # lava in the (4,5) square
rewards[(19,20)] = lava_reward # lava in the (19,20) square
rewards[(4,3)] = charging_reward   # charging station square

# setup set of absorbing states -
absorbing_state_set = Set{Tuple{Int,Int}}()
for (k,v) ∈ rewards
    push!(absorbing_state_set, k);
end

Now we build the world itself. [The `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/) takes the dimensions and the rewards and returns a `world::MyRectangularGridWorldModel` instance holding the coordinate system and the moves available from each square:

In [ ]:
world = build(MyRectangularGridWorldModel,
    (nrows = number_of_rows, ncols = number_of_cols, rewards = rewards));

### The world the robot cannot see
[The `build_mdp(...)` function](src/Utility.jl) builds the reward matrix $R(s,a)$ and the transition array $T(s^{\prime}\,|\,s,a)$ with the same rules we used for value iteration. Entering the charging station earns 100, entering a lava pit costs 1000, an ordinary move costs 1, and a move off the grid costs 50000 and leaves the robot where it is. Every move is certain. It returns the `m::MyMDPProblemModel` instance:

In [ ]:
m = build_mdp(world, rewards, absorbing_state_set; γ = γ); # the MDP: the states, actions, T, R, and γ

Value iteration reads $R$ and $T$ directly, but the Q-learning agent never does. It learns through [the `lavaworld(...)` function](src/Utility.jl) instead. Given a state $s$ and an action $a$, the function returns the next state $s^{\prime}$ and the reward $r$, as if the robot had made the move. It draws the next state at random from $T(s^{\prime}\,|\,s,a)$, so it takes a random number generator as its last argument.

Let's try one move: from the square $(20,20)$, move left, which is action 1:

In [ ]:
let
    s = world.states[(20,20)]; # the state of the square (20,20)
    (s′, r) = lavaworld(m, s, 1, Xoshiro(1)); # move left; Xoshiro(1) is a random number generator with seed 1
    println("the robot moved from ", world.coordinates[s], " to ", world.coordinates[s′], " and collected ", r)
end

The square to the left of $(20,20)$ is the lava pit at $(19,20)$, so this one move costs the robot 1000 and would end its episode. The robot learns where the lava pits are only by entering them.

Finally, the agent needs to know which states end an episode. We collect their state numbers in the `absorbing_states::Set{Int64}` set:

In [ ]:
absorbing_states = Set{Int64}(); # the states that end an episode
for coordinate ∈ absorbing_state_set
    push!(absorbing_states, world.states[coordinate]);
end

### Things to think about

__Question:__ The robot is never told where the lava pits are. How does it find out that a lava pit is bad, and what does finding out cost a real robot? Briefly explain.

___

## Task 2: Implement Q-Learning
In this task, we implement Q-learning in the `myqlearning(...)` function, train the robot, and check our code on a small problem.

Q-learning estimates the action-value function $Q(s,a)$ from the moves the robot makes. After each move, it updates one entry of the table:

$$
\begin{equation*}
Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)
\end{equation*}
$$

where $0<\alpha\leq 1$ is the learning rate and $\gamma$ is the discount factor. The term $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$ is the temporal difference target, and its difference from $Q(s,a)$ is the temporal difference error.

__Initialize__: Given the state space $\mathcal{S}$, the action space $\mathcal{A}$, the absorbing states, a discount factor $0\leq\gamma<1$, a learning rate $0<\alpha\leq 1$, an exploration probability $0\leq\epsilon\leq 1$, a number of episodes $N\geq 1$, and a maximum number of steps per episode $d\geq 1$, set $Q(s,a)\gets 0$ for all $s\in\mathcal{S}$ and $a\in\mathcal{A}$.

For $i = 1$ to $N$ __do__:
1. Choose a starting state $s$ that is not absorbing, uniformly at random, and set the step counter $t\gets 0$.
2. While $s$ is not absorbing and $t<d$ __do__:
    - Choose the action $a$: roll a random number $p\in[0,1]$.
        - If $p\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q(s,a^{\prime})$ (exploitation).
    - Take action $a$, and observe the reward $r$ and the next state $s^{\prime}$ from the world.
    - Update the action value with the temporal difference error:
      $$Q(s,a)\gets Q(s,a)+\alpha\left(r+\gamma\max_{a^{\prime}\in\mathcal{A}}Q(s^{\prime},a^{\prime})-Q(s,a)\right)$$
    - Update the state $s\gets s^{\prime}$ and the step counter $t\gets t+1$.

__Extract Policy__: For each state $s\in\mathcal{S}$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q(s,a)$.

We store the agent in a `MyQLearningAgentModel` instance from the course package. [The `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/) takes the states, the actions, the learning rate $\alpha$, the discount factor $\gamma$, and the starting table $Q$, all zeros.

Let's set the training constants and build the agent:

In [ ]:
α = 0.5; # learning rate (you can change this)
ϵ = 0.1; # exploration probability (you can change this)
number_of_episodes = 2000; # number of episodes N (you can change this)
maxsteps = 500; # maximum number of steps per episode d
agent = build(MyQLearningAgentModel, (
    states = 𝒮, # the states
    actions = 𝒜, # the actions
    α = α, # the learning rate
    γ = γ, # the discount factor
    Q = zeros(nstates, nactions), # the starting table: all zeros
));

### Implement Our Version
Open [`src/Compute.jl`](src/Compute.jl) and complete [the `myqlearning(...)` function](src/Compute.jl). A TODO comment marks each of the three lines you write:

> __Implementation tasks__
>
> * __TODO 1:__ The greedy action, the exploitation branch of the epsilon-greedy rule.
> * __TODO 2:__ The temporal difference target.
> * __TODO 3:__ The update of $Q(s,a)$.

Save the file, re-run the setup cell to reload your edits, then train the agent. The untouched function throws an error.

Our version returns a `NamedTuple` holding the learned table `Q` and the counts `visits`, where `visits[s,a]` is the number of times the robot took action `a` in state `s`. Every random draw comes from a generator seeded with `seed`, so a rerun gives the same result.

__Prediction:__ After 2000 episodes, will the learned policy lead the robot from the square $(20,20)$ to the charging station: {yes, by a shortest path; yes, by a longer path; or no}?

In [ ]:
result = L12bQLearning.myqlearning(agent, m; worldmodel = lavaworld, absorbing = absorbing_states,
    episodes = number_of_episodes, maxsteps = maxsteps, ϵ = ϵ, seed = 5800); # the untouched stub throws an error
println("moves made during training: ", sum(result.visits))

### Check: Your Q-Learning
We check our version on a small problem with three states and two actions. In state 1, action 1 moves to state 2 and earns $-1$, and action 2 stays in state 1 and earns $-2$. In state 2, action 1 moves to state 3 and earns $10$, and action 2 moves back to state 1 and earns $-1$. State 3 is absorbing. With $\gamma=0.9$, the optimal action values are $Q^{*}(2,1)=10$, $Q^{*}(1,1)=-1+0.9\cdot 10=8$, $Q^{*}(1,2)=-2+0.9\cdot 8=5.2$, and $Q^{*}(2,2)=-1+0.9\cdot 8=6.2$.

Three things must hold:
* __One step by hand:__ We also make state 2 absorbing, so every episode starts in state 1, and we take one step with $\epsilon=0$. Starting from $Q(1,1)=0$ and $Q(1,2)=4$, the greedy action is 2, and an agent with $\alpha=0.5$ and $\gamma=0.5$ gets $Q(1,2)=4+0.5\left(-2+0.5\cdot 4-4\right)=2$. Starting from $Q(1,1)=6$ and $Q(1,2)=4$, the greedy action is 1, and an agent with $\alpha=0.25$ gets $Q(1,1)=6+0.25\left(-1+0-6\right)=4.25$, because state 2 is absorbing.
* __The optimal values:__ Every move here is certain, so with $\alpha=1$ each update replaces $Q(s,a)$ with its target. Enough random episodes then give the optimal action values exactly.
* __Absorbing rows stay zero:__ An episode ends as soon as the robot enters an absorbing state, so the grid world rows of the charging station and the lava pits are never visited or updated.

Let's run the checks:

In [ ]:
@testset "My Q-learning" begin

    # the three-state problem: T[s, s′, a] and R[s, a] -
    T_small = zeros(3, 3, 2);
    R_small = zeros(3, 2);
    T_small[1, 2, 1] = 1.0; R_small[1, 1] = -1.0; # state 1, action 1: to state 2, reward -1
    T_small[1, 1, 2] = 1.0; R_small[1, 2] = -2.0; # state 1, action 2: stay in state 1, reward -2
    T_small[2, 3, 1] = 1.0; R_small[2, 1] = 10.0; # state 2, action 1: to state 3, reward 10
    T_small[2, 1, 2] = 1.0; R_small[2, 2] = -1.0; # state 2, action 2: back to state 1, reward -1
    T_small[3, 3, 1] = 1.0; T_small[3, 3, 2] = 1.0; # state 3 is absorbing: we stay put
    small = build(MyMDPProblemModel, (𝒮 = [1, 2, 3], 𝒜 = [1, 2], T = T_small, R = R_small, γ = 0.9));

    # one step by hand: the greedy action is 2, and the update gives 2 -
    one_step_agent = build(MyQLearningAgentModel, (states = [1, 2, 3], actions = [1, 2],
        α = 0.5, γ = 0.5, Q = [0.0 4.0; 0.0 0.0; 0.0 0.0]));
    one_step = L12bQLearning.myqlearning(one_step_agent, small; worldmodel = lavaworld,
        absorbing = Set([2, 3]), episodes = 1, maxsteps = 1, ϵ = 0.0);
    @test one_step.visits == [0 1; 0 0; 0 0] # one move: action 2 in state 1
    @test one_step.Q ≈ [0.0 2.0; 0.0 0.0; 0.0 0.0]

    # one step by hand: the greedy action is 1, and the update gives 4.25 -
    other_agent = build(MyQLearningAgentModel, (states = [1, 2, 3], actions = [1, 2],
        α = 0.25, γ = 0.5, Q = [6.0 4.0; 0.0 0.0; 0.0 0.0]));
    other_step = L12bQLearning.myqlearning(other_agent, small; worldmodel = lavaworld,
        absorbing = Set([2, 3]), episodes = 1, maxsteps = 1, ϵ = 0.0);
    @test other_step.visits == [1 0; 0 0; 0 0] # one move: action 1 in state 1
    @test other_step.Q ≈ [4.25 4.0; 0.0 0.0; 0.0 0.0]

    # random episodes with α = 1 give the optimal action values -
    small_agent = build(MyQLearningAgentModel, (states = [1, 2, 3], actions = [1, 2],
        α = 1.0, γ = 0.9, Q = zeros(3, 2)));
    small_result = L12bQLearning.myqlearning(small_agent, small; worldmodel = lavaworld,
        absorbing = Set([3]), episodes = 500, maxsteps = 50, ϵ = 1.0);
    @test small_result.Q ≈ [8.0 5.2; 10.0 6.2; 0.0 0.0]

    # the rows of the absorbing states of the grid world are never visited or updated -
    for s ∈ absorbing_states
        @test all(result.Q[s, :] .== 0.0)
        @test all(result.visits[s, :] .== 0)
    end
end;

___

## Task 3: Compare with Value Iteration
In this task, we compare the learned policy with the optimal policy. Value iteration knows $R$ and $T$, so it computes the optimal values directly. [The `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.solve-Tuple{MyValueIterationModel,%20MyMDPProblemModel}) runs value iteration, [the `Q(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.Q-Tuple{MyMDPProblemModel,%20Vector{Float64}}) turns the optimal values into the optimal action values $Q^{*}(s,a)$, and [the `policy(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/mdp/#VLDataScienceMachineLearningPackage.policy-Tuple{Matrix{Float64}}) reads the best action from each row of a table.

Let's compute the optimal policy and read the learned one:

In [ ]:
solution = solve(MyValueIterationModel(1000), m); # value iteration, at most 1000 sweeps
Qstar = Q(m, solution.U); # the optimal action values Q*(s,a), one row per state
π_star = policy(Qstar); # the optimal policy
π_q = policy(result.Q); # the learned policy

### Count the optimal moves
In a square where several moves tie for the best, any of them is optimal. [The `count_optimal_moves(...)` function](src/Utility.jl) counts a learned move as optimal when its optimal action value $Q^{*}(s,\pi_{q}(s))$ is within a small tolerance of the best value $\max_{a}Q^{*}(s,a)$, where $\pi_{q}$ is the learned policy.

Let's count the optimal moves, and compare the learned and optimal values of the square $(20,20)$:

In [ ]:
(number_optimal, number_of_squares) = count_optimal_moves(Qstar, π_q, absorbing_states);
println("the learned move is optimal in ", number_optimal, " of ", number_of_squares, " squares")

s_start = world.states[(20,20)]; # the state of the square (20,20)
println("value of (20,20): learned ", round(maximum(result.Q[s_start, :]), digits = 2),
    ", optimal ", round(solution.U[s_start], digits = 2))

### Visualize the learned policy
Now let's follow the learned policy from a starting square, and compare it with the optimal policy from the same square. Specify a `startstate::Tuple{Int,Int}` as an $(x, y)$ coordinate tuple:

In [ ]:
startstate = (20,20); # where to start the path simulation (you can change this)

[The `follow_policy(...)` function](src/Utility.jl) starts at a square and applies the policy one move at a time, until the robot enters an absorbing state, tries to leave the grid, or returns to a square it has already visited, which means it would walk in a loop forever:

In [ ]:
# walk each policy from the start square -
(path, outcome) = follow_policy(π_q, startstate, world, rewards, absorbing_state_set;
    charging_reward = charging_reward);
(path_star, outcome_star) = follow_policy(π_star, startstate, world, rewards, absorbing_state_set;
    charging_reward = charging_reward);

println("learned policy: the robot ", outcome, " after ", length(path) - 1, " moves")
println("optimal policy: the robot ", outcome_star, " after ", length(path_star) - 1, " moves")

Let's draw the learned policy and its path. [The `draw_path(...)` function](src/Utility.jl) marks the charging station in green and the lava pits in red. A gray arrow in every other square shows the move the learned policy chooses there, and the robot's path is the blue line from the blue dot:

In [ ]:
# draw the grid, the learned policy, and its path -
draw_path(path, π_q, world, rewards, absorbing_state_set;
    charging_reward = charging_reward, lava_reward = lava_reward,
    number_of_rows = number_of_rows, number_of_cols = number_of_cols)

__What do we see?__ The learned policy takes the robot from $(20,20)$ to the charging station in as few moves as the optimal policy. Yet the learned move is optimal in most squares but not all of them, and the learned value of the start square is still well below its optimal value. The greedy policy needs only the best move in each square to have the largest value, not the correct value, so the policy can be right before the values are.

__Prediction:__ Suppose we train a new agent for only 200 episodes. Will its policy still lead the robot from `startstate` to the charging station: {yes, by a shortest path; yes, by a longer path; or no}?

In [ ]:
let

    # train a new agent with different settings (try other values) -
    episodes_new = 200; # number of episodes
    ϵ_new = 0.1; # exploration probability
    agent_new = build(MyQLearningAgentModel, (states = 𝒮, actions = 𝒜, α = α, γ = γ,
        Q = zeros(nstates, nactions)));
    result_new = L12bQLearning.myqlearning(agent_new, m; worldmodel = lavaworld, absorbing = absorbing_states,
        episodes = episodes_new, maxsteps = maxsteps, ϵ = ϵ_new, seed = 5800);
    π_new = policy(result_new.Q); # the new learned policy

    # count the optimal moves, and follow the new policy -
    (optimal_new, squares_new) = count_optimal_moves(Qstar, π_new, absorbing_states);
    (path_new, outcome_new) = follow_policy(π_new, startstate, world, rewards, absorbing_state_set;
        charging_reward = charging_reward);
    println(episodes_new, " episodes with ϵ = ", ϵ_new, ": ", sum(result_new.visits), " moves of training")
    println("the learned move is optimal in ", optimal_new, " of ", squares_new, " squares")
    println("the robot ", outcome_new, " after ", length(path_new) - 1, " moves")

    draw_path(path_new, π_new, world, rewards, absorbing_state_set;
        charging_reward = charging_reward, lava_reward = lava_reward,
        number_of_rows = number_of_rows, number_of_cols = number_of_cols) # return
end

__What do we see?__ After 200 episodes, the policy no longer leads the robot to the charging station: from the start square, it walks in a loop. Each update carries the station's reward back by only one square, from a square to the square before it. After 200 episodes, the reward has spread only to the squares near the station, and far from it the values still reflect only the cost of moving.

### Things to think about

__Question:__ Set `episodes_new = 2000` and `ϵ_new = 0.0`, so the robot never explores, and rerun the cell. Does the robot still learn a path to the charging station? Every move costs at least 1 and the table starts at zero, so what happens to $Q(s,a)$ the first time the robot makes a move, and how does that push a greedy robot to try other moves? Briefly explain.

__Question:__ Now set `ϵ_new = 1.0`, so every move is random. In how many squares is the learned move optimal, and how many moves did training take compared with $\epsilon=0.1$? The robot never followed its learned policy during training, so which term of the update lets it learn the best moves while it moves at random? Briefly explain.

___

## Summary
In this lab, we implemented Q-learning, trained a robot to navigate the lava grid world from its own moves, and compared the policy it learned with the optimal policy from value iteration.

> __Key Takeaways:__
>
> * __Q-learning needs only the moves the robot makes:__ Our robot learned from the next square and the reward of each move, which a simulator returned, without ever reading the rewards or the transition probabilities. One update per move, toward the temporal difference target, was enough to learn a shortest path to the charging station.
>
> * __The policy can be right before the values are:__ After 2000 episodes, the learned policy matched the optimal path from the start square, although the learned values were still below the optimal values. The greedy policy needs only the best move in each square to have the largest value.
>
> * __The amount of training decides how far the reward spreads:__ Each update carries the charging station's reward back by one square, so with too few episodes it never reached the start square and the policy walked in a loop. Longer training, or more exploration, spread the reward over more of the grid.

We can now change the number of episodes, the exploration probability, or the learning rate and see how the learned policy responds.

___